### import libraries

In [43]:
import os
import yaml

import numpy as np
import matplotlib.pyplot as plt
from XLO_sim.XLO_sim import XLO_sim
from XLO_sim.Plot import XLO_plot
from XLO_sim import tools

import numpy.fft as fft

import scipy.constants as sp_const
au_in_eV = sp_const.value('atomic unit of energy') / sp_const.value('atomic unit of charge')

import multiprocessing as mp
import shutil

notebook_path = os.path.abspath("__file__")
notebook_directory = os.path.dirname(notebook_path)
base_directory = os.path.dirname(notebook_directory)

### Transmission

In [44]:
tpad = 1000
xpad = 64
ypad = 64

hwKalpha1N = 8047.91

In [45]:
def yaml_modify_seed_energy_and_target_energy(input_yaml_path, output_yaml_path, new_seed_energy, target_energy_eV):
    # Read the original YAML file
    with open(input_yaml_path, 'r') as file:
        yaml_data = yaml.safe_load(file)

    yaml_data['E_seed_uJ'] = new_seed_energy
    yaml_data['monochromator_target_energy_eV'] = float(target_energy_eV)

    # Write the modified data to a new YAML file
    with open(output_yaml_path, 'w') as file:
        yaml.safe_dump(yaml_data, file)

    print(f"Modified YAML file saved to {output_yaml_path}")

In [46]:
ar_Eseed_values = [40] #[200, 150, 100, 70, 40, 20, 10, 5, 1, 0.1]

# Absolute target photon energies to scan (eV), spanning +/-15 eV around the
# Cu Kalpha1 line (hwKalpha1N, read from the base config below).
with open(base_directory + '/config/base/Cu-seed-mono-SASE.yaml', 'r') as file:
    hwKalpha1N = yaml.safe_load(file)['hwKalpha1N']
ar_energy_values = hwKalpha1N + np.arange(-15, 16, 3, dtype=float)

ar_yaml = []

for Eseed in ar_Eseed_values:
    generated_directory = base_directory + '/config/generated'
    generated_path = os.path.join(generated_directory)
    if not os.path.exists(generated_path):
        os.makedirs(generated_path)

    for target_energy_eV in ar_energy_values:
        input_yaml_path = base_directory + '/config/base/Cu-seed-mono-SASE.yaml'
        output_yaml_path = base_directory + '/config/generated/Cu-seed-mono-SASE_' + f'{Eseed:.2f}' + 'uJ_' + f'{target_energy_eV:.2f}' + 'eV.yaml'
        yaml_modify_seed_energy_and_target_energy(input_yaml_path, output_yaml_path, Eseed, target_energy_eV)
        ar_yaml.append(output_yaml_path)

Modified YAML file saved to /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-mono-SASE_40.00uJ_8032.91eV.yaml
Modified YAML file saved to /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-mono-SASE_40.00uJ_8035.91eV.yaml
Modified YAML file saved to /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-mono-SASE_40.00uJ_8038.91eV.yaml
Modified YAML file saved to /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-mono-SASE_40.00uJ_8041.91eV.yaml
Modified YAML file saved to /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-mono-SASE_40.00uJ_8044.91eV.yaml
Modified YAML file saved to /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-mono-SASE_40.00uJ_8047.91eV.yaml
Modified YAML file sav

### calculating for given amount of repetitions for each yaml

In [47]:
nrep = 1

# Get the number of available CPU cores
num_cpus = mp.cpu_count()

# Set the maximum number of processes to use
num_processes = num_cpus - 1  # Leave one core for system processes or other tasks
print("num_processes", num_processes)


data_path = os.path.join(base_directory, 'data/' + np.datetime_as_string(np.datetime64('now')))
if not os.path.exists(data_path):
    os.makedirs(data_path)

def run_simulation(yaml, run_path, rep):
    print('repetition ', rep + 1)

    X = XLO_sim(yaml)
    X.random_seed = rep  # Set the random seed for reproducibility
    seed_field = tools.Ocelot_SASE_seed_111_dcm_pstxy(X)
    X.configure(seed_field)
    X.run_3D()

    womega_ar, I_int_thy_w_0, I_thy0_w_0 = tools.SF_spectrum_w(X, 0, ypad, tpad)
    womega_ar, I_int_thy_w_last, I_thy0_w_last = tools.SF_spectrum_w(X, -1, ypad, tpad)

    target_energy_eV = X.monochromator_target_energy_eV
    date_string = np.datetime_as_string(np.datetime64('now'))
    np.savez_compressed(
        os.path.join(run_path, f"run_at_seed_{X.E_seed_uJ:.1f}_uJ__energy_{target_energy_eV:.2f}_eV__repetition_{rep + 1}_{date_string}.npz"),
        target_energy_eV=target_energy_eV,
        womega_ar=womega_ar,
        I_int_thy_w_0=I_int_thy_w_0,
        I_thy0_w_0=I_thy0_w_0,
        I_int_thy_w_last=I_int_thy_w_last,
        I_thy0_w_last=I_thy0_w_last
    )

def run_repetitions_from_yaml(yaml, nrep, num_processes, data_path):
    date_string = np.datetime_as_string(np.datetime64('now'))
    X = XLO_sim(yaml)
    target_energy_eV = X.monochromator_target_energy_eV

    run_path = os.path.join(data_path, f'runs_seed_{X.E_seed_uJ:.1f}_uJ__energy_{target_energy_eV:.2f}_eV_{date_string}')
    if not os.path.exists(run_path):
        os.makedirs(run_path)

    shutil.copy2(yaml, run_path)

    # Force the 'fork' start method explicitly so behavior is consistent
    # between macOS (default 'spawn' since Python 3.8) and the Linux
    # cluster (default 'fork'). 'spawn' fails here with AttributeError
    # because it re-imports __main__ to find run_simulation, which does
    # not work for functions defined in a notebook.
    ctx = mp.get_context('fork')
    with ctx.Pool(processes=num_processes) as pool:
        pool.starmap(run_simulation, [(yaml, run_path, rep) for rep in range(nrep)])

if __name__ == '__main__':
    for yaml_value in ar_yaml:
        print(f"Running simulations for YAML file: {yaml_value}")
        run_repetitions_from_yaml(yaml_value, nrep, num_processes, data_path)

num_processes 7
Running simulations for YAML file: /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-mono-SASE_40.00uJ_8032.91eV.yaml
repetition  1
8032.91
8032.91
Running simulations for YAML file: /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-mono-SASE_40.00uJ_8035.91eV.yaml
repetition  1
8035.91
8035.91
Running simulations for YAML file: /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-mono-SASE_40.00uJ_8038.91eV.yaml
repetition  1
8038.91
8038.91
Running simulations for YAML file: /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-mono-SASE_40.00uJ_8041.91eV.yaml
repetition  1
8041.91
8041.91
Running simulations for YAML file: /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-mono-SASE_40.00uJ_8044.91eV.yaml
repetition  1
